# PRAGMA-lite masked-transaction training

This notebook trains the current end-to-end model: **Event Encoder → Profile Encoder → History Encoder → masked-field MLM head**. It uses the account-disjoint train/validation parquet splits created in the exploration notebook.

## Before running in a Colab runtime

The VS Code Colab extension runs code on a remote Colab machine; it cannot see this Windows folder by itself. The recommended setup is to put the project on the Colab VM's fast, temporary `/content` disk, not to train from mounted Drive. The setup cell can clone a public Git repository, or you can upload one project zip directly from your laptop.

- `finBERTlitemodules/`
- `data/processed/czech_bank/`
- `financial_db_Teradata/`

The project material needed for this experiment is about 71 MiB, so a direct zip upload is perfectly reasonable. It does **not** consume Google Drive storage, but it must be repeated after the Colab VM resets. In Colab, select a GPU runtime before you run the notebook. The configuration cell is the one place to change batch size, epochs, or a resume checkpoint.

In [ ]:
# Install the one dependency Colab may not already have. This installs
# into the current temporary VM, not into Google Drive.
%pip install -q pyarrow

import shutil
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
PROJECT_ROOT = Path('/content/FinancialBertForTransactions') if IN_COLAB else Path.cwd()

# OPTION A — recommended after you push this repository to GitHub. Set the
# URL once, then the code is cloned onto the Colab VM at each new session.
REPOSITORY_URL = ''  # e.g. 'https://github.com/YOUR_USERNAME/FinancialBertForTransactions.git'
if IN_COLAB and REPOSITORY_URL and not PROJECT_ROOT.exists():
    !git clone --depth=1 {REPOSITORY_URL} {PROJECT_ROOT}

# OPTION B — no Drive and no GitHub needed. First zip this project on your
# laptop, then change this to True and choose that zip in the upload dialog.
# The zip must unpack to /content/FinancialBertForTransactions/.
UPLOAD_PROJECT_ZIP = False
if IN_COLAB and UPLOAD_PROJECT_ZIP and not PROJECT_ROOT.exists():
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Choose exactly one project zip archive.')
    archive_path = Path(next(iter(uploaded)))
    shutil.unpack_archive(archive_path, '/content')

PROJECT_ROOT = PROJECT_ROOT.resolve()
assert (PROJECT_ROOT / 'finBERTlitemodules').exists(), (
    f'Cannot find the project at {PROJECT_ROOT}. Use Option A or Option B above.'
)
# This makes `from finBERTlitemodules import ...` work exactly as it does
# locally: Python looks for packages relative to the project root.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')
print(f'Running in Colab: {IN_COLAB}')
if IN_COLAB:
    !nvidia-smi

## Imports and configuration

The split is by `account_id`, not transaction row. Tokenizer boundaries are fitted **only** from the train split and then reused for validation. `MAX_EVENTS` is the history context window; events before that window are excluded from the current sample.

In [ ]:
from dataclasses import asdict
from time import perf_counter

import pandas as pd
import torch
from torch import Tensor
from torch.nn import functional as F
from torch.utils.data import DataLoader

from finBERTlitemodules import (
    EventMLMDemoModel,
    FinBERTLiteCzechDataset,
    TransformerConfig,
    apply_value_mlm_mask,
    collate_account_records,
    fit_tokenizer_bundle,
    save_tokenizer_bundle,
)

SEED = 42
EPOCHS = 5
BATCH_SIZE = 32          # Start here on a Colab T4/L4; increase only after one epoch works.
MAX_EVENTS = 64
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-2
GRAD_CLIP_NORM = 1.0
NUM_WORKERS = 2 if torch.cuda.is_available() else 0

# Set to e.g. CHECKPOINT_DIR / 'last.pt' to continue an interrupted run.
RESUME_FROM = None

DATA_DIR = PROJECT_ROOT / 'data' / 'processed' / 'czech_bank'
SOURCE_DIR = PROJECT_ROOT / 'financial_db_Teradata'
CHECKPOINT_DIR = PROJECT_ROOT / 'checkpoints' / 'pragma_lite_mlm'
TOKENIZER_DIR = CHECKPOINT_DIR / 'tokenizers'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_EVENTS = DATA_DIR / 'events_train.parquet'
TRAIN_PROFILES = DATA_DIR / 'profile_train.parquet'
VALID_EVENTS = DATA_DIR / 'events_valid.parquet'
VALID_PROFILES = DATA_DIR / 'profile_valid.parquet'
for required_path in (TRAIN_EVENTS, TRAIN_PROFILES, VALID_EVENTS, VALID_PROFILES, SOURCE_DIR):
    assert required_path.exists(), f'Missing required path: {required_path}'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_ENABLED = DEVICE.type == 'cuda'
torch.manual_seed(SEED)
if DEVICE.type == 'cuda':
    torch.cuda.manual_seed_all(SEED)

print(f'Device: {DEVICE}; mixed precision: {AMP_ENABLED}')

## Fit the train-only tokenizer bundle and build datasets

This happens once at the beginning of a fresh run. The resulting tokenizer states are saved with the checkpoint so validation and a future resumed run use identical numeric bucket boundaries.

In [ ]:
# `fit_tokenizer_bundle` may take a moment because it reads the train split
# and the lifetime source tables. It never reads validation data to fit bins.
train_events_frame = pd.read_parquet(TRAIN_EVENTS)
train_profiles_frame = pd.read_parquet(TRAIN_PROFILES)
tokenizers = fit_tokenizer_bundle(train_events_frame, train_profiles_frame, SOURCE_DIR)
save_tokenizer_bundle(tokenizers, TOKENIZER_DIR)
vocabulary_size = len(tokenizers.event.token_to_id)
del train_events_frame, train_profiles_frame

train_dataset = FinBERTLiteCzechDataset(
    TRAIN_EVENTS, TRAIN_PROFILES, SOURCE_DIR, tokenizers,
    max_events=MAX_EVENTS, random_cutoff=True, seed=SEED,
)
valid_dataset = FinBERTLiteCzechDataset(
    VALID_EVENTS, VALID_PROFILES, SOURCE_DIR, tokenizers,
    max_events=MAX_EVENTS, random_cutoff=False, seed=SEED,
)

loader_kwargs = {
    'batch_size': BATCH_SIZE,
    'num_workers': NUM_WORKERS,
    'pin_memory': DEVICE.type == 'cuda',
    'collate_fn': collate_account_records,
}
train_loader = DataLoader(
    train_dataset, shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
    **loader_kwargs,
)
valid_loader = DataLoader(valid_dataset, shuffle=False, **loader_kwargs)

print(f'Train accounts: {len(train_dataset):,}')
print(f'Valid accounts: {len(valid_dataset):,}')
print(f'Vocabulary size: {vocabulary_size}')
print(f'Train batches/epoch: {len(train_loader):,}')

## Create the model and the training helpers

The MLM labels are the original values at selected field positions; all other labels are `-100` and are ignored by cross-entropy. The event keys remain visible, so the prediction head knows which field's candidate distribution it is modelling.

In [ ]:
MODEL_CONFIG = TransformerConfig(
    d_model=64,
    num_heads=4,
    num_layers=2,
    ffn_dim=128,
    dropout=0.1,
)
model = EventMLMDemoModel(vocabulary_size, MODEL_CONFIG).to(DEVICE)
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler(DEVICE.type, enabled=AMP_ENABLED)

def move_tensors(batch: dict[str, object]) -> dict[str, object]:
    return {
        name: value.to(DEVICE, non_blocking=True) if isinstance(value, Tensor) else value
        for name, value in batch.items()
    }

def forward_from_batch(masked_batch: dict[str, object]):
    return model(
        masked_batch['event_key_ids'],
        masked_batch['event_value_ids'],
        masked_batch['event_mask'],
        masked_batch['profile_key_ids'],
        masked_batch['profile_value_ids'],
        masked_batch['profile_mask'],
        masked_batch['profile_rope_time'],
        masked_batch['history_rope_time'],
        masked_batch['calendar_features'],
    )

def mlm_cross_entropy(logits: Tensor, labels: Tensor) -> Tensor:
    return F.cross_entropy(
        logits.reshape(-1, logits.shape[-1]),
        labels.reshape(-1),
        ignore_index=-100,
    )

def make_mask_generator(seed: int) -> torch.Generator:
    # `apply_value_mlm_mask` samples directly on the batch's device.
    return torch.Generator(device=DEVICE.type).manual_seed(seed)

print(f'Model parameters: {sum(parameter.numel() for parameter in model.parameters()):,}')

## Optional: resume an interrupted run

Set `RESUME_FROM = CHECKPOINT_DIR / 'last.pt'` in the configuration cell, then run this cell before training. The same tokenizer state saved in `TOKENIZER_DIR` should accompany the checkpoint.

In [ ]:
start_epoch = 1
best_validation_loss = float('inf')
if RESUME_FROM is not None:
    checkpoint = torch.load(RESUME_FROM, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    start_epoch = int(checkpoint['epoch']) + 1
    best_validation_loss = float(checkpoint['best_validation_loss'])
    print(f'Resumed after epoch {start_epoch - 1}; best validation loss={best_validation_loss:.4f}')
else:
    print('Starting a fresh run.')

## Train and validate

Every training epoch uses deterministic but different random cutoff days through `train_dataset.set_epoch(epoch)`. Validation always uses each account's final cutoff day. The validation mask is reset to a fixed seed each epoch, so its loss is comparable over time.

In [ ]:
history: list[dict[str, float]] = []
train_mask_generator = make_mask_generator(SEED + 100)

for epoch in range(start_epoch, EPOCHS + 1):
    epoch_start = perf_counter()
    train_dataset.set_epoch(epoch)
    model.train()
    train_loss_sum = 0.0
    train_target_count = 0

    for raw_batch in train_loader:
        batch = move_tensors(raw_batch)
        masked_batch = apply_value_mlm_mask(batch, generator=train_mask_generator)
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP_ENABLED):
            output = forward_from_batch(masked_batch)
            loss = mlm_cross_entropy(output.logits, masked_batch['mlm_labels'])

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()

        target_count = int(masked_batch['mlm_mask'].sum().item())
        train_loss_sum += loss.item() * target_count
        train_target_count += target_count

    model.eval()
    valid_loss_sum = 0.0
    valid_target_count = 0
    valid_mask_generator = make_mask_generator(SEED + 200)
    with torch.no_grad():
        for raw_batch in valid_loader:
            batch = move_tensors(raw_batch)
            masked_batch = apply_value_mlm_mask(batch, generator=valid_mask_generator)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=AMP_ENABLED):
                output = forward_from_batch(masked_batch)
                loss = mlm_cross_entropy(output.logits, masked_batch['mlm_labels'])
            target_count = int(masked_batch['mlm_mask'].sum().item())
            valid_loss_sum += loss.item() * target_count
            valid_target_count += target_count

    if train_target_count == 0 or valid_target_count == 0:
        raise RuntimeError('No MLM targets were selected; check masking probabilities.')
    train_loss = train_loss_sum / train_target_count
    valid_loss = valid_loss_sum / valid_target_count
    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step()
    elapsed_seconds = perf_counter() - epoch_start

    epoch_record = {
        'epoch': float(epoch),
        'train_loss': train_loss,
        'validation_loss': valid_loss,
        'learning_rate': current_lr,
        'seconds': elapsed_seconds,
    }
    history.append(epoch_record)
    print(
        f"epoch {epoch:>2}/{EPOCHS}: train={train_loss:.4f}, valid={valid_loss:.4f}, "
        f"lr={current_lr:.2e}, time={elapsed_seconds:.1f}s"
    )

    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_validation_loss': min(best_validation_loss, valid_loss),
        'model_config': asdict(MODEL_CONFIG),
        'vocabulary_size': vocabulary_size,
        'max_events': MAX_EVENTS,
    }
    torch.save(checkpoint, CHECKPOINT_DIR / 'last.pt')
    if valid_loss < best_validation_loss:
        best_validation_loss = valid_loss
        torch.save(checkpoint, CHECKPOINT_DIR / 'best.pt')
        print('  saved new best.pt')

metrics = pd.DataFrame(history)
metrics.to_csv(CHECKPOINT_DIR / 'metrics.csv', index=False)
metrics

## Inspect the learning curve and saved artifacts

MLM loss should fall meaningfully from its initial value. Do not treat it as a downstream credit-risk or recommendation metric; this is representation pre-training only.

In [ ]:
import matplotlib.pyplot as plt

ax = metrics.set_index('epoch')[['train_loss', 'validation_loss']].plot(
    marker='o', figsize=(8, 4), grid=True,
)
ax.set_ylabel('masked-field cross-entropy')
ax.set_title('PRAGMA-lite MLM training')
plt.show()

print('Saved artifacts:')
for path in sorted(CHECKPOINT_DIR.rglob('*')):
    if path.is_file():
        print(' ', path.relative_to(PROJECT_ROOT))

## Sensible first experiments

1. Confirm a 1-epoch GPU run completes and `best.pt` is written.
2. Increase `EPOCHS` to 20–50 only after the validation curve is stable.
3. Increase `BATCH_SIZE` gradually if GPU memory has room.
4. Keep `MAX_EVENTS=64` initially; changing it changes the effective task context.
5. Do not refit tokenizers on validation/test data or overwrite the tokenizer files attached to a checkpoint.

The next separate piece of work, once MLM behaviour is credible, is a downstream task head and a leakage-safe task-specific split.